In [ ]:
# --- Importação de Bibliotecas Padrão e de Terceiros ---
import os
import sys


# --- Configuração do Caminho do Projeto para Importações Locais ---
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest

In [26]:
df = pd.read_csv(r"C:\repositorio\public-data-analysis-anomaly-detection-ml\data\processed\notas_fiscais.csv",sep=";",encoding="utf-8")

# Preparação dos Dados

In [27]:
# --- 1. Tratamento de valores faltantes ---
# A coluna 'EVENTOS' possui 160 valores nulos. Preencher com 'Sem Evento'.
df["EVENTOS"] = df["EVENTOS"].fillna("Sem Evento")
print("Valores nulos após tratamento (EVENTOS):\n")
print(df.isnull().sum().to_markdown(numalign="left", stralign="left"))

# --- 2. Feature Engineering ---
# Converter a coluna 'DATA' para datetime
df["DATA"] = pd.to_datetime(df["DATA"])

Valores nulos após tratamento (EVENTOS):

|               | 0   |
|:--------------|:----|
| ID            | 0   |
| ORGAO         | 0   |
| FORNECEDOR    | 0   |
| CNPJ          | 0   |
| MUNICIPIO     | 0   |
| VALOR_NF      | 0   |
| ITENS         | 0   |
| TIPO_EVENTO   | 0   |
| DATA          | 0   |
| EVENTOS       | 0   |
| CHAVE_NF      | 0   |
| MUNICIPIO_MUN | 0   |
| UF            | 0   |
| POPULACAO     | 0   |
| COD_IBGE      | 0   |
| ANO_MES       | 0   |


In [28]:
# Extrair features temporais
df["ANO"] = df["DATA"].dt.year
df["MES"] = df["DATA"].dt.month
df["DIA_SEMANA"] = df["DATA"].dt.dayofweek  # 0=Segunda, 6=Domingo
df["DIA_MES"] = df["DATA"].dt.day
df = df.drop('ITENS', axis=1)
df = df.drop('EVENTOS', axis=1)

# Criar feature de valor por população (VALOR_NF_POR_POPULACAO)
# Tratar casos onde POPULACAO pode ser zero para evitar divisão por zero
df["VALOR_NF_POR_POPULACAO"] = df["VALOR_NF"] / df["POPULACAO"].replace(0, np.nan) # Substituir 0 por NaN para depois preencher ou dropar
df["VALOR_NF_POR_POPULACAO"] = df["VALOR_NF_POR_POPULACAO"].fillna(0) # Preencher NaNs resultantes da divisão por zero com 0

print("\n### Primeiras 5 linhas do dataset com novas features:\n")
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))


### Primeiras 5 linhas do dataset com novas features:

| ID   | ORGAO                                             | FORNECEDOR                                   | CNPJ           | MUNICIPIO             | VALOR_NF   | TIPO_EVENTO        | DATA                | CHAVE_NF                                     | MUNICIPIO_MUN         | UF   | POPULACAO   | COD_IBGE   | ANO_MES   | ANO   | MES   | DIA_SEMANA   | DIA_MES   | VALOR_NF_POR_POPULACAO   |
|:-----|:--------------------------------------------------|:---------------------------------------------|:---------------|:----------------------|:-----------|:-------------------|:--------------------|:---------------------------------------------|:----------------------|:-----|:------------|:-----------|:----------|:------|:------|:-------------|:----------|:-------------------------|
| 1950 | Ministério da Saúde - Unidades com vínculo direto | RCA PRODUTOS E SERVICOS LTDA                 | 69207850000161 | SANTA BARBARA D'OESTE | 16004.5    

In [29]:
# --- 3. Normalização/Padronização e Codificação de Variáveis Categóricas ---
# Identificar colunas numéricas e categóricas para o pré-processamento
numeric_features = ["VALOR_NF", "POPULACAO", "VALOR_NF_POR_POPULACAO", "ANO", "MES", "DIA_SEMANA", "DIA_MES"]
# Colunas categóricas para One-Hot Encoding. Excluindo 'ID', 'CNPJ', 'CHAVE_NF' pois são identificadores únicos e 'DATA', 'ANO_MES' (já extraídas).
categorical_features = ["FORNECEDOR", "MUNICIPIO", "MUNICIPIO_MUN", "UF"]

In [30]:
# Criar pré-processador usando ColumnTransformer
# Para colunas numéricas, usar StandardScaler
# Para colunas categóricas, usar OneHotEncoder (handle_unknown='ignore' para evitar erros com categorias novas)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [31]:
# Aplicar o pré-processamento e converter a matriz esparsa para densa
X_processed = preprocessor.fit_transform(df).toarray()

# Obter nomes das colunas após One-Hot Encoding
onehot_feature_names = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)
processed_feature_names = numeric_features + list(onehot_feature_names)

In [32]:
df_processed = pd.DataFrame(X_processed, columns=processed_feature_names)

In [33]:
df_original_for_output = df.copy()

# Iniciando o Machine Learning

In [34]:
# Garantir que ambos os dataframes tenham o mesmo número de linhas
if df_processed.shape[0] != df_original_for_output.shape[0]:
    print("ERRO: O número de linhas entre o dataframe processado e o original não corresponde.")
    # Uma possível causa é a remoção de linhas devido a valores nulos em alguma etapa.
    # Vamos re-alinhar os índices para garantir a correspondência.
    # No entanto, a melhor abordagem é garantir que a preparação dos dados não remova linhas inesperadamente.
    # Por agora, vamos parar a execução para evitar resultados incorretos.
    exit()


In [35]:
# --- Treinar e aplicar o modelo Isolation Forest ---
# Definir o modelo Isolation Forest
contamination_rate = 0.05 # 1% de anomalias esperadas

print(f"Treinando Isolation Forest com contamination_rate = {contamination_rate}")

Treinando Isolation Forest com contamination_rate = 0.05


In [36]:
model = IsolationForest(n_estimators=100, contamination=contamination_rate, random_state=42, n_jobs=-1)

# Treinar o modelo
model.fit(df_processed)

,n_estimators,100
,max_samples,'auto'
,contamination,0.05
,max_features,1.0
,bootstrap,False
,n_jobs,-1
,random_state,42
,verbose,0
,warm_start,False


In [37]:
# Prever anomalias (-1 para anomalias, 1 para inliers)
anomaly_predictions = model.predict(df_processed)

In [38]:
# Obter os scores de decisão (quanto menor, mais anômalo)
anomaly_scores = model.decision_function(df_processed)

In [39]:
# Adicionar as previsões de anomalia e os scores de decisão ao DataFrame original
df_original_for_output["ANOMALY_PREDICTION"] = anomaly_predictions
df_original_for_output["ANOMALY_SCORE"] = anomaly_scores

In [40]:
# Mapear as previsões para "Sim" ou "Não"
df_original_for_output["POSSIVEL_ANOMALIA"] = df_original_for_output["ANOMALY_PREDICTION"].apply(lambda x: "Sim" if x == -1 else "Não")

print("\n### Contagem de Anomalias Detectadas:\n")
print(df_original_for_output["POSSIVEL_ANOMALIA"].value_counts().to_markdown(numalign="left", stralign="left"))

print("\n### Primeiras 5 linhas com rótulo de anomalia:\n")
print(df_original_for_output[df_original_for_output["POSSIVEL_ANOMALIA"] == "Sim"].head().to_markdown(index=False, numalign="left", stralign="left"))


### Contagem de Anomalias Detectadas:

| POSSIVEL_ANOMALIA   | count   |
|:--------------------|:--------|
| Não                 | 267933  |
| Sim                 | 14102   |

### Primeiras 5 linhas com rótulo de anomalia:

| ID   | ORGAO                                             | FORNECEDOR                           | CNPJ           | MUNICIPIO          | VALOR_NF   | TIPO_EVENTO        | DATA                | CHAVE_NF                                     | MUNICIPIO_MUN      | UF   | POPULACAO   | COD_IBGE   | ANO_MES   | ANO   | MES   | DIA_SEMANA   | DIA_MES   | VALOR_NF_POR_POPULACAO   | ANOMALY_PREDICTION   | ANOMALY_SCORE   | POSSIVEL_ANOMALIA   |
|:-----|:--------------------------------------------------|:-------------------------------------|:---------------|:-------------------|:-----------|:-------------------|:--------------------|:---------------------------------------------|:-------------------|:-----|:------------|:-----------|:----------|:------|:------|:----------

In [41]:
# Definir diretório de saída (pasta processed dentro de data)
output_dir = os.path.join(project_root, "data", "processed_isolation_forest")

# Salvar o dataset com as anomalias detectadas para a próxima fase
df_original_for_output.to_csv(os.path.join(output_dir,"notas_fiscais_com_anomalias.csv"), index=False, sep=";", encoding="utf-8")
print("Dataset com anomalias salvo como notas_fiscais_com_anomalias.csv")

Dataset com anomalias salvo como notas_fiscais_com_anomalias.csv


In [23]:
# Salvar o modelo treinado 
output_dir = os.path.join(project_root, "model")

joblib.dump(model,os.path.join(output_dir, "isolation_forest_model.joblib") )
print("Modelo Isolation Forest salvo como isolation_forest_model.joblib")

# Salvar os nomes das features usadas no modelo (para interpretação posterior)
with open(os.path.join(output_dir,"model_features.txt"), "w") as f:
    for feature in df_processed.columns:
        f.write(feature + "\n")
print("Nomes das features usadas no modelo salvas em model_features.txt")

Modelo Isolation Forest salvo como isolation_forest_model.joblib
Nomes das features usadas no modelo salvas em model_features.txt
